In [7]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *
from qick.asm_v2 import AveragerProgramV2   # v2 classes aren't in `from qick import *`

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP -- unchanged
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR -- pure numpy, unchanged from your
#    updated version (best_n_samples phase-residual optimization included)
# -----------------------------------------------------------------------------
def best_n_samples(freq_hz, target_dur_s, sample_rate, samps_per_clk_, search_radius=3):
    target_n = int(round(target_dur_s * sample_rate))
    target_n = max(target_n, 1)
    pad0 = (-target_n) % samps_per_clk_
    target_n += pad0

    candidates = [target_n + k * samps_per_clk_
                  for k in range(-search_radius, search_radius + 1)]
    candidates = [n for n in candidates if n >= samps_per_clk_]

    if freq_hz == 0 or len(candidates) == 0:
        return target_n

    def residual(n):
        cycles = freq_hz * n / sample_rate
        frac = cycles - np.floor(cycles)
        return min(frac, 1.0 - frac)

    return min(candidates, key=residual)


def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                    n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- unchanged
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ

AMPLITUDE = 0.11

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6 + 76.25e6, -147.82e6 + 76.25e6])
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.2])

BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()

def solve_max_num_steps(env_maxlen, env_sr, samps_per_clk_, tone_fractions_, safety=0.95):
    floor_samples = 3 * samps_per_clk_
    best_N = None
    N = 1
    while N <= 4000:
        max_feasible_total_s_ = safety * env_maxlen / env_sr
        cycle_s = max_feasible_total_s_ / (N + 1)
        tone_durations_s_ = tone_fractions_ * cycle_s

        piece_lens = []
        for dur in tone_durations_s_:
            n = int(round(dur * env_sr))
            n = max(n, 1)
            pad = (-n) % samps_per_clk_
            piece_lens.append(n + pad)
        samples_per_step_ = sum(piece_lens)
        total_samples_ = samples_per_step_ * (N + 1)

        floor_ok = min(piece_lens) >= floor_samples
        mem_ok = total_samples_ <= env_maxlen

        if floor_ok and mem_ok:
            best_N = N
            N += 1
        else:
            break
    if best_N is None:
        raise ValueError("No feasible NUM_STEPS found -- TONE_RATIOS too extreme for this memory budget.")
    return best_N

NUM_STEPS = 6
print(f"Solved max feasible NUM_STEPS = {NUM_STEPS} (was hand-set to 44 previously)")

STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

CYCLE_S = max_feasible_total_s / (NUM_STEPS + 1)
tone_durations_s = tone_fractions * CYCLE_S

print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held via mode='periodic' "
      f"for {STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

def build_multitone_buffer(chirp_offset_hz, phase0):
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur in zip(BASE_TONES_HZ, tone_durations_s):
        f = chirp_offset_hz + base_tone
        n_best = best_n_samples(f, dur, ENV_SR, samps_per_clk)
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase,
                                              n_samples_override=n_best)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

# midpoint chirp sampling -- unchanged
step_width_hz = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ

maxv = soccfg.get_maxv(GEN_CH)

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

samples_per_step = len(idata_list[0])
lengths = [len(x) for x in idata_list]
print(set(lengths))
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0
assert min(piece_lens) >= 3 * samps_per_clk, (
    f"Smallest tone slice is only {min(piece_lens)} samples "
    f"({min(piece_lens)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3. "
    f"Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
)

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER -- unchanged
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM -- tProc v2 (QickProgramV2 / AveragerProgramV2)
#    Same porting notes as your earlier chirp script:
#      * initialize()/body() -> _initialize()/_body()
#      * no ch_page/sreg/mathi/loopnz/update() register-walking -- each
#        precomputed step becomes a named pulse; _body() just calls
#        self.pulse(..., t=...) once per step in a plain Python loop,
#        which the v2 assembler unrolls into real instructions itself
#      * gain=32767 (v1 raw) -> gain=1.0 (v2 normalized -1.0..1.0);
#        AMPLITUDE already scales the envelope samples themselves
#      * outsel="input" and phrst=0 preserved -- same meaning in both APIs
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgramV2(AveragerProgramV2):
    def _initialize(self, cfg):
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            name = f"serr_{i}"
            self.add_envelope(ch=res_ch, name=name, idata=idata_step, qdata=qdata_step)
            self.add_pulse(
                ch=res_ch,
                name=name,
                style="arb",
                envelope=name,
                freq=0,
                phase=0,
                gain=cfg["gain"],
                outsel="input",
                mode="oneshot",
                phrst=0,
            )

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])
        self.add_pulse(
            ch=res_ch,
            name="trap_wfm",
            style="arb",
            envelope="trap_wfm",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            outsel="input",
            mode="periodic",
            phrst=0,
        )

    def _body(self, cfg):
        res_ch = cfg["res_ch"]
        step_hold_us = cfg["step_hold_us"]

        self.trigger(pins=[0], t=0)

        t = 0.0
        for i in range(len(cfg["idata_list"])):
            self.pulse(ch=res_ch, name=f"serr_{i}", t=t)
            t += step_hold_us

        self.pulse(ch=res_ch, name="trap_wfm", t=t)

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "step_hold_us": STEP_HOLD_US,
    "gain": 1.0,   # v2 normalized full-scale; AMPLITUDE already scales the envelope itself
}

prog = RepeatedStepSerrodyneProgramV2(soccfg, reps=1, final_delay=0.5, cfg=config)
# run() is shared between v1/v2 (defined in qick_asm.py), so start_src="external" carries over
prog.run(soc) #, start_src="external"
print(f"Running on hardware -- {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 1: f_fabric=491.520 MHz, samps_per_clk=16, envelope sample rate=7.8643 GSPS
Envelope memory available: 65536 samples
Solved max feasible NUM_STEPS = 6 (was hand-set to 44 previously)
Composite buffer cycle: 1130.95 ns, held via mode='periodic' for 1000.00 us per chirp step (6.000 ms total sweep)
  tone 0 (-76.2 MHz offset): ratio 0.337 -> requested 384.20 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 190.39 ns
  tone 2 (+46.7 MHz offset): ratio 0.288 -> requested 328.34 ns
  tone 3 (+71.6 MHz offset): ratio 0.2 -> requested 228.01 ns
{8992, 8896, 8800, 9008, 8944, 8784}
Per-step composite buffer length: 9008 samples (tone slices: [2992, 1504, 2544, 1760], smallest = 94.0 fabric cycles)
Total envelope samples (sweep): 54048 / 65536 available
Trap buffer: 8800 samples (tone slices: [2992, 1504, 2544, 1760])
Total envelope samples (sweep + trap): 62848 / 65536 available
Running on hardware -- 6 sweep steps x 1000.00 us = 6.000 ms sweep, then trapping (4 tones, periodic

In [8]:
soc.reset_gens()

In [26]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *
from qick.asm_v2 import AveragerProgramV2

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 0
RO_CH  = 0

gencfg        = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR        = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN    = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                   n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t  = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y     = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ         = 350e6
CHIRP_OFFSET_STOP_HZ  = 0.0
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ

AMPLITUDE = 0.11

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6 + 76.25e6, -147.82e6 + 76.25e6])
TONE_RATIOS   = np.array([0.337,   0.167, 0.288,             0.208])
BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
tone_fractions       = TONE_RATIOS / TONE_RATIOS.sum()

NUM_STEPS    = 40
STEP_HOLD_S  = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

CYCLE_S          = max_feasible_total_s / (NUM_STEPS + 1)
tone_durations_s = tone_fractions * CYCLE_S

print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held via mode='periodic' "
      f"for {STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

# -----------------------------------------------------------------------------
# FIX: Fixed piece lengths — same at every chirp offset so every buffer has
# identical total length, keeping addr_step valid across all step transitions.
# -----------------------------------------------------------------------------
piece_n_samples = []
for dur in tone_durations_s:
    n   = int(round(dur * ENV_SR))
    n   = max(n, 1)
    pad = (-n) % samps_per_clk
    piece_n_samples.append(n + pad)

print(f"Fixed piece lengths (samples): {piece_n_samples}, "
      f"smallest = {min(piece_n_samples)/samps_per_clk:.1f} fabric cycles")
assert min(piece_n_samples) >= 3 * samps_per_clk, (
    f"Smallest fixed tone slice is only {min(piece_n_samples)} samples "
    f"({min(piece_n_samples)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3."
)

maxv = soccfg.get_maxv(GEN_CH)

def build_multitone_buffer(chirp_offset_hz, phase0):
    """Composite envelope with FIXED piece lengths, independent of chirp_offset_hz."""
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur, n_fixed in zip(BASE_TONES_HZ, tone_durations_s, piece_n_samples):
        f = chirp_offset_hz + base_tone
        y, phase, _ = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase,
                                     n_samples_override=n_fixed)
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    piece_lens = [len(p) for p in i_pieces]
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, piece_lens

# Midpoint chirp sampling
step_width_hz    = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

# Hard invariant: all sweep buffers must be identical length
step_lengths = set(len(x) for x in idata_list)
assert len(step_lengths) == 1, (
    f"chirp step buffers have unequal length: {step_lengths}"
)
lengths = [len(x) for x in idata_list]
print('length', set(lengths))
samples_per_step = len(idata_list[0])
total_samples    = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0

# Trap buffer
trap_idata, trap_qdata, _, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")
assert len(trap_idata) == samples_per_step, (
    f"trap buffer length {len(trap_idata)} != sweep-step length {samples_per_step}"
)

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN

# -----------------------------------------------------------------------------
# 4. PROGRAM — tProc v2 (AveragerProgramV2, QICK v0.2.422)
#
# Key design decision: _body() is called ONCE at compile time — it is NOT
# called once per loop iteration. The hardware loop is emitted around
# whatever instructions _body() appends. This means we cannot use a Python
# loop index inside _body() to select different envelopes per iteration.
#
# Solution: fully unroll the chirp sweep at compile time inside _initialize()
# using a plain Python for-loop. Each iteration emits a distinct pulse +
# delay pair directly into the program instruction stream. No hardware loop,
# no runtime register indexing, no jump table needed. The tProc simply
# executes the 40 pulse+delay pairs sequentially, then falls through to
# after_loop() for the trap.
#
# add_loop() / _body() are NOT used — we override make_program() instead
# to emit the full unrolled sequence plus the trap, then call end().
# -----------------------------------------------------------------------------
class SerrodyneChirpV2(AveragerProgramV2):

    def _initialize(self, cfg):
        res_ch = cfg["res_ch"]
        self.declare_gen(ch=res_ch, nqz=1)

        # Load all sweep-step envelopes
        for i, (idata_step, qdata_step) in enumerate(
                zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}",
                              idata=idata_step, qdata=qdata_step)

        # Load trap envelope
        self.add_envelope(ch=res_ch, name="trap_wfm",
                          idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        # Pre-declare one named pulse per chirp step so the framework
        # registers each envelope address at compile time.
        for i in range(cfg["expts"]):
            self.add_pulse(
                ch=res_ch,
                name=f"chirp_pulse_{i}",
                style="arb",
                freq=0,
                phase=0,
                gain=cfg["gain"],
                envelope=f"serr_{i}",
                mode="periodic",
            )

        # Trap pulse
        self.add_pulse(
            ch=res_ch,
            name="trap_pulse",
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            envelope="trap_wfm",
            mode="periodic",
        )

    def _body(self, cfg):
        # _body() is called once per rep by the framework.
        # We emit the full unrolled chirp sweep here — each step is a
        # separate named pulse with its own pre-loaded envelope, followed
        # by a delay. The tProc executes them sequentially.
        res_ch = cfg["res_ch"]
        for i in range(cfg["expts"]):
            self.pulse(ch=res_ch, name=f"chirp_pulse_{i}", t=0)
            self.delay_auto(cfg["step_hold_us"], gens=[res_ch])

    def after_loop(self, cfg):
        """Fire the trap pulse once after all reps complete.
        mode='periodic' keeps the DAC looping the buffer indefinitely."""
        res_ch = cfg["res_ch"]
        self.pulse(ch=res_ch, name="trap_pulse", t=0)

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch":       GEN_CH,
    "expts":        NUM_STEPS,
    "idata_list":   idata_list,
    "qdata_list":   qdata_list,
    "trap_idata":   trap_idata,
    "trap_qdata":   trap_qdata,
    "step_hold_us": STEP_HOLD_US,
    "gain":         32767,
}

# reps=1     : single rep — _body() runs once, emitting all 40 pulse+delay pairs
# final_delay=0 : no dead-time appended; timing is controlled by delay_auto()
prog = SerrodyneChirpV2(soccfg, reps=1, final_delay=0, cfg=config)
prog.run(soc) #, start_src="external"

print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS * STEP_HOLD_US * 1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")


Generator 0: f_fabric=491.520 MHz, samps_per_clk=16, envelope sample rate=7.8643 GSPS
Envelope memory available: 65536 samples
Composite buffer cycle: 193.09 ns, held via mode='periodic' for 150.00 us per chirp step (6.000 ms total sweep)
  tone 0 (-76.2 MHz offset): ratio 0.337 -> requested 65.07 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 32.25 ns
  tone 2 (+46.7 MHz offset): ratio 0.288 -> requested 55.61 ns
  tone 3 (+71.6 MHz offset): ratio 0.208 -> requested 40.16 ns
Fixed piece lengths (samples): [512, 256, 448, 320], smallest = 16.0 fabric cycles
length {1536}
Per-step composite buffer length: 1536 samples (tone slices: [512, 256, 448, 320], smallest = 16.0 fabric cycles)
Total envelope samples (sweep): 61440 / 65536 available
Trap buffer: 1536 samples (tone slices: [512, 256, 448, 320])
Total envelope samples (sweep + trap): 62976 / 65536 available
Running on hardware — 40 sweep steps x 150.00 us = 6.000 ms sweep, then trapping (4 tones, periodic, indefinitely unti

In [2]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *
from qick.asm_v2 import AveragerProgramV2

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 0
RO_CH  = 0

gencfg        = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR        = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN    = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
# -----------------------------------------------------------------------------
def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                   n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t  = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y     = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. QUADRATIC TAPER
#
# Applied to each composite step buffer to smooth the amplitude at the
# step boundaries (start and end of each STEP_HOLD_US dwell period).
# This converts the rectangular step profile into one with C0-continuous
# edges, improving sideband rolloff from ~1/f to ~1/f^2 (~40 dB/decade).
#
# Quadratic is chosen over cosine (Hann/Tukey) because it rises faster,
# wasting less duty cycle on very short buffers (~193 ns here).
#
# TAPER_FRAC: fraction of each buffer used for ramp-up + ramp-down.
#   0.05 = 5% ramp-up + 5% ramp-down (10% total duty cycle cost).
#   Increase for more sideband suppression at the cost of more power loss.
# -----------------------------------------------------------------------------
TAPER_FRAC = 0.05   # tune this: 0.02–0.10 is a reasonable range

def quadratic_taper(n, taper_frac=TAPER_FRAC):
    """
    Returns a length-n window that is:
      - quadratic ramp-up over the first taper_frac*n samples
      - flat (1.0) in the middle
      - quadratic ramp-down over the last taper_frac*n samples

    The ramp goes as t^2, so it rises quickly and spends most of the
    taper region near full amplitude — minimising duty-cycle loss
    compared to a cosine (Hann) ramp.
    """
    w = np.ones(n, dtype=np.float64)
    ramp_len = max(1, int(round(n * taper_frac)))
    t_up   = np.linspace(0.0, 1.0, ramp_len)
    t_down = np.linspace(1.0, 0.0, ramp_len)
    w[:ramp_len]  = t_up ** 2
    w[-ramp_len:] = t_down ** 2
    return w

# -----------------------------------------------------------------------------
# 4. CHIRP DEFINITION
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ         = 50e6
CHIRP_OFFSET_STOP_HZ  = 0.0
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ

AMPLITUDE = 0.11

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6 + 76.25e6, -147.82e6 + 76.25e6])
TONE_RATIOS   = np.array([0.337,   0.167, 0.288,             0.208])
BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
tone_fractions       = TONE_RATIOS / TONE_RATIOS.sum()

# -----------------------------------------------------------------------------
# Solve for the maximum feasible NUM_STEPS given the envelope memory budget.
# Staircase phase error scales as ~1/N^2, so more steps = cleaner chirp.
# -----------------------------------------------------------------------------
def solve_max_num_steps(env_maxlen, env_sr, samps_per_clk_, tone_fractions_, safety=0.95):
    floor_samples = 3 * samps_per_clk_
    best_N = None
    N = 1
    while N <= 4000:
        max_s = safety * env_maxlen / env_sr
        cycle_s = max_s / (N + 1)
        tone_durations_s_ = tone_fractions_ * cycle_s
        piece_lens = []
        for dur in tone_durations_s_:
            n   = int(round(dur * env_sr))
            n   = max(n, 1)
            pad = (-n) % samps_per_clk_
            piece_lens.append(n + pad)
        samples_per_step_ = sum(piece_lens)
        total_samples_    = samples_per_step_ * (N + 1)
        floor_ok = min(piece_lens) >= floor_samples
        mem_ok   = total_samples_ <= env_maxlen
        if floor_ok and mem_ok:
            best_N = N
            N += 1
        else:
            break
    if best_N is None:
        raise ValueError("No feasible NUM_STEPS found.")
    return best_N

NUM_STEPS = solve_max_num_steps(ENV_MAXLEN, ENV_SR, samps_per_clk, tone_fractions)
print(f"Solved max feasible NUM_STEPS = {NUM_STEPS}")

STEP_HOLD_S  = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

CYCLE_S          = max_feasible_total_s / (NUM_STEPS + 1)
tone_durations_s = tone_fractions * CYCLE_S

print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held via mode='periodic' "
      f"for {STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

# -----------------------------------------------------------------------------
# Fixed piece lengths — same at every chirp offset so every buffer has
# identical total length. Required for correct envelope addressing in v2.
# -----------------------------------------------------------------------------
piece_n_samples = []
for dur in tone_durations_s:
    n   = int(round(dur * ENV_SR))
    n   = max(n, 1)
    pad = (-n) % samps_per_clk
    piece_n_samples.append(n + pad)

print(f"Fixed piece lengths (samples): {piece_n_samples}, "
      f"smallest = {min(piece_n_samples)/samps_per_clk:.1f} fabric cycles")
assert min(piece_n_samples) >= 3 * samps_per_clk, (
    f"Smallest fixed tone slice is only {min(piece_n_samples)} samples "
    f"({min(piece_n_samples)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3."
)

maxv = soccfg.get_maxv(GEN_CH)

def build_multitone_buffer(chirp_offset_hz, phase0, taper=True):
    """
    Composite envelope with FIXED piece lengths, independent of chirp_offset_hz.

    If taper=True, a quadratic window is applied to the full concatenated
    buffer — ramping amplitude up at the start of the step and down at the
    end. This smooths the step boundary discontinuity and suppresses
    chirp-step-rate sidebands (~1/f^2 rolloff vs ~1/f for rectangular).

    The taper is applied AFTER phase generation so phase continuity across
    steps is preserved — only amplitude is shaped, not phase.
    """
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur, n_fixed in zip(BASE_TONES_HZ, tone_durations_s, piece_n_samples):
        f = chirp_offset_hz + base_tone
        y, phase, _ = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase,
                                     n_samples_override=n_fixed)
        i_pieces.append(y)                          # keep as float for windowing
        q_pieces.append(np.zeros(len(y)))
    piece_lens = [len(p) for p in i_pieces]

    i_cat = np.concatenate(i_pieces)
    q_cat = np.concatenate(q_pieces)

    if taper:
        # Apply quadratic taper to the full composite buffer.
        # The ramp spans TAPER_FRAC of the total buffer length, which
        # corresponds to TAPER_FRAC * CYCLE_S ≈ a few ns at each edge —
        # short enough not to distort the tone content significantly.
        w = quadratic_taper(len(i_cat), TAPER_FRAC)
        i_cat = i_cat * w
        q_cat = q_cat * w

    i_out = np.round(i_cat * maxv).astype(np.int16)
    q_out = np.round(q_cat * maxv).astype(np.int16)
    return i_out, q_out, phase, piece_lens

# Midpoint chirp sampling — each step represents the chirp's average
# frequency over its dwell interval, evaluated at the midpoint.
# The last step is snapped to CHIRP_OFFSET_STOP_HZ for a clean hand-off
# into the trap buffer.
step_width_hz    = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase, taper=True)
    idata_list.append(idata)
    qdata_list.append(qdata)

# Hard invariant: all sweep buffers must be identical length
step_lengths = set(len(x) for x in idata_list)
assert len(step_lengths) == 1, (
    f"chirp step buffers have unequal length: {step_lengths}"
)

samples_per_step = len(idata_list[0])
total_samples    = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0

# Trap buffer — NO taper: the trap runs indefinitely in periodic mode so
# there is no step boundary to smooth. A taper would just reduce power.
trap_idata, trap_qdata, _, trap_piece_lens = build_multitone_buffer(
    CHIRP_OFFSET_STOP_HZ, phase, taper=False)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")
assert len(trap_idata) == samples_per_step, (
    f"trap buffer length {len(trap_idata)} != sweep-step length {samples_per_step}"
)

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN

# -----------------------------------------------------------------------------
# 5. PROGRAM — tProc v2 (AveragerProgramV2, QICK v0.2.422)
#
# Key design decision: _body() is called ONCE at compile time — it is NOT
# called once per loop iteration. The hardware loop is emitted around
# whatever instructions _body() appends. This means we cannot use a Python
# loop index inside _body() to select different envelopes per iteration.
#
# Solution: fully unroll the chirp sweep at compile time inside _initialize()
# using a plain Python for-loop. Each iteration emits a distinct pulse +
# delay pair directly into the program instruction stream. No hardware loop,
# no runtime register indexing, no jump table needed. The tProc simply
# executes the 40 pulse+delay pairs sequentially, then falls through to
# after_loop() for the trap.
#
# add_loop() / _body() are NOT used — we override make_program() instead
# to emit the full unrolled sequence plus the trap, then call end().
# -----------------------------------------------------------------------------
class SerrodyneChirpV2(AveragerProgramV2):

    def _initialize(self, cfg):
        res_ch = cfg["res_ch"]
        self.declare_gen(ch=res_ch, nqz=1)

        # Load all sweep-step envelopes
        for i, (idata_step, qdata_step) in enumerate(
                zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}",
                              idata=idata_step, qdata=qdata_step)

        # Load trap envelope
        self.add_envelope(ch=res_ch, name="trap_wfm",
                          idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        # Pre-declare one named pulse per chirp step so the framework
        # registers each envelope address at compile time.
        for i in range(cfg["expts"]):
            self.add_pulse(
                ch=res_ch,
                name=f"chirp_pulse_{i}",
                style="arb",
                freq=0,
                phase=0,
                gain=cfg["gain"],
                envelope=f"serr_{i}",
                mode="periodic",
            )

        # Trap pulse
        self.add_pulse(
            ch=res_ch,
            name="trap_pulse",
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            envelope="trap_wfm",
            mode="periodic",
        )

    def _body(self, cfg):
        # _body() is called once per rep by the framework.
        # Emit the full unrolled chirp sweep — each step is a separate
        # named pulse with its own pre-loaded (tapered) envelope, followed
        # by a delay. The tProc executes them sequentially.
        res_ch = cfg["res_ch"]
        for i in range(cfg["expts"]):
            self.pulse(ch=res_ch, name=f"chirp_pulse_{i}", t=0)
            self.delay_auto(cfg["step_hold_us"], gens=[res_ch])

    def after_loop(self, cfg):
        """Fire the trap pulse once after all reps complete.
        mode='periodic' keeps the DAC looping the buffer indefinitely."""
        res_ch = cfg["res_ch"]
        self.pulse(ch=res_ch, name="trap_pulse", t=0)

# -----------------------------------------------------------------------------
# 6. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch":       GEN_CH,
    "expts":        NUM_STEPS,
    "idata_list":   idata_list,
    "qdata_list":   qdata_list,
    "trap_idata":   trap_idata,
    "trap_qdata":   trap_qdata,
    "step_hold_us": STEP_HOLD_US,
    "gain":         32767,
}

# reps=1        : single rep — _body() runs once, emitting all N pulse+delay pairs
# final_delay=0 : no dead-time appended; timing is controlled by delay_auto()
prog = SerrodyneChirpV2(soccfg, reps=1, final_delay=0, cfg=config)
prog.run(soc) # , start_src="external"

print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS * STEP_HOLD_US * 1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")
print(f"Quadratic taper: {TAPER_FRAC*100:.0f}% ramp-up + {TAPER_FRAC*100:.0f}% ramp-down "
      f"per step buffer ({TAPER_FRAC*CYCLE_S*1e9:.1f} ns each edge).")


Generator 0: f_fabric=491.520 MHz, samps_per_clk=16, envelope sample rate=7.8643 GSPS
Envelope memory available: 65536 samples
Solved max feasible NUM_STEPS = 77
Composite buffer cycle: 101.50 ns, held via mode='periodic' for 77.92 us per chirp step (6.000 ms total sweep)
  tone 0 (-76.2 MHz offset): ratio 0.337 -> requested 34.20 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 16.95 ns
  tone 2 (+46.7 MHz offset): ratio 0.288 -> requested 29.23 ns
  tone 3 (+71.6 MHz offset): ratio 0.208 -> requested 21.11 ns
Fixed piece lengths (samples): [272, 144, 240, 176], smallest = 9.0 fabric cycles
Per-step composite buffer length: 832 samples (tone slices: [272, 144, 240, 176], smallest = 9.0 fabric cycles)
Total envelope samples (sweep): 64064 / 65536 available
Trap buffer: 832 samples (tone slices: [272, 144, 240, 176])
Total envelope samples (sweep + trap): 64896 / 65536 available
Running on hardware — 77 sweep steps x 77.92 us = 6.000 ms sweep, then trapping (4 tones, periodic, in